# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedheshatthi104ff-hub/Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Which content pages should a human reviewer prioritize for a potential refresh, based on historical search-performance signals and the likelihood of a decline in future impressions?

The project supports a practical decision: ranking pages so that limited content-review resources can be directed toward pages that may need attention. The output is a ranked review queue with a risk score, reason code, and suggested action. It is intended for human decision support, not automated editing or causal prediction.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data

- **Dataset:** FlyRank pseudonymized warehouse release `flyrank_pseudonymized_warehouse_release_v20260703`, accessed through the gated Hugging Face dataset.
- **Table used:** `fact_content_daily_performance`.
- **Data fields:** Daily search impressions, clicks, and average position aggregated by client, content page, and date.
- **Development dataset:** Features from January 1, 2026 to March 31, 2026, with the future outcome measured from April 1 to April 30, 2026. This dataset contained 100,849 pages from 44 clients.
- **Sealed test dataset:** Features from March 3, 2026 to May 31, 2026, with the outcome measured from June 1 to June 30, 2026. This contained 111,247 pages from 50 clients.
- **Filtering:** Pages were included only if they had at least 100 impressions during the final 30 days of the feature window.
- **Exclusions:** Product scores and flags, future-dated information, client names, URLs, search queries, and page titles were excluded to avoid circular results, leakage, and privacy risks.

The data was pseudonymized and used at page-client level. No raw client identities, page URLs, or private search queries were used in the modeling.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Methodology

**Label definition:** A page was labeled as a decline if its impressions during the following 30 days were less than 80% of its impressions during the preceding 30 days. This represents an observed decline of at least 20%.

**Features:** Nine historical features were used:
1. Impressions in the last 30 days.
2. Impressions in the preceding 60 days.
3. Total impressions over 90 days.
4. Total clicks over 90 days.
5. Click-through rate (CTR).
6. Impression-weighted average position.
7. Number of active days.
8. Standard deviation of daily impressions.
9. Trend ratio comparing recent impressions with the earlier daily rate.

**Baseline:** A transparent rule-based score was constructed using the percentile rank of recent demand and the size of the recent impression drop. The score was calculated as 0.4 × demand percentile + 0.6 × recent drop.

**Models:**
- Logistic Regression with standardized features.
- Random Forest with 300 trees and a minimum leaf size of 5.

**Validation:** Five-fold GroupKFold cross-validation was used, holding out entire clients rather than randomly splitting individual pages. This reduces the risk of client-level leakage.

**Metrics:** Average Precision (primary metric), ROC AUC, Precision@K and lift over the base rate, with K = 50, 200 and 1,000.

**Leakage audit:** The feature window ended before the outcome window began. Product scores and future information were excluded. A deliberately added future-derived leakage feature raised AUC to 1.0, confirming that the audit could detect this type of leakage. The leakage feature was not included in the reported models. Client-level validation and a frozen, single-scoring final test were also used.

Client-relative features were tested and dropped because they reduced development average precision for the Random Forest from 0.557 to 0.527.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Results

**Development results — 5-fold client-grouped cross-validation**

| Method | ROC AUC | Average Precision | Precision@50 |
|---|---:|---:|---:|
| Rule baseline | 0.475 | 0.483 | 0.62 |
| Logistic Regression | 0.548 | 0.511 | 0.86 |
| Random Forest | 0.588 | 0.557 | 0.68 |

The Random Forest performed best overall in development average precision and ROC AUC. The baseline performed below chance in ROC AUC. Precision@50 varied considerably between folds, so it was treated as a noisy metric.

**Sealed June test results**

| Method | ROC AUC | Average Precision | P@50 | P@200 | P@1000 |
|---|---:|---:|---:|---:|---:|
| Rule baseline | 0.520 | 0.671 | 0.82 | 0.795 | 0.766 |
| Logistic Regression | 0.596 | 0.702 | 0.72 | 0.645 | 0.700 |
| Random Forest | 0.630 | 0.721 | 0.66 | 0.705 | 0.757 |

The Random Forest achieved the highest ROC AUC and average precision on the sealed June test. Its average precision of 0.721 was approximately 12% above the June base rate of 0.643.

The Random Forest was consistently better than the rule baseline across client-level resampling, but its improvement over Logistic Regression could not be statistically distinguished from zero. The simple rule also matched or outperformed both learned models among the top 50 to 1,000 pages.

These results suggest that the model offers modest ranking value across the wider page inventory, while the highest-ranked pages should still be manually checked.

## 5. Limitations

*What this work cannot claim.*

### Limitations

1. **Observational data:** The model identifies patterns associated with future impression declines. It does not demonstrate that refreshing a page causes recovery.
2. **Noisy decline label:** A decline in impressions may reflect seasonality, natural regression after a peak, changing demand, or noise rather than a genuine content problem.
3. **Limited predictive signal:** The improvement in average precision over the June base rate was modest.
4. **Top-ranked uncertainty:** Precision among the highest-ranked pages was not consistently better than the transparent rule baseline, particularly at small K.
5. **Client concentration:** A few large clients account for a substantial proportion of the development dataset, limiting generalizability.
6. **Limited time coverage:** The evaluation uses one development period and one sealed test month, so longer-term stability has not been established.
7. **Policy choices:** The 20% decline threshold and 100-impression minimum are project-specific choices and are not universal definitions of content decline.
8. **Leakage audit limitations:** The checklist and deliberate-leak test help detect specific forms of leakage, but cannot prove that every possible leakage source has been eliminated.

The system should therefore be used as a prioritization aid for human review rather than an automated content-editing decision system.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Ranked Recommendations

The model generates a ranked queue of content pages for human review.

| Reason Code | Recommended Action | When to Use | Potential Risk |
|---|---|---|---|
| `declining_with_demand` | Review for refresh | The page has meaningful impression volume and its recent impressions are already below its earlier level. | The decline may be seasonal, caused by demand shifting to another page, or may not require an update. |
| `high_model_risk` | Monitor | The model assigns high risk, but there is no clearly visible decline yet. | Low recent volume or a sudden increase in impressions can make the score noisy. |
| Comparison with baseline | Compare model score with the rule-based score | When only a limited number of pages can be reviewed, use the transparent rule as a second reference. | The rule may outperform the model at the very top of the ranking. |

**Action playbook:**

1. Rank pages using the Random Forest risk score.
2. Inspect the reason code and recent impression trend.
3. Prioritize pages with an actual recent decline and meaningful impression volume.
4. Monitor pages with high model risk but no confirmed decline.
5. For the top few dozen pages, verify the underlying performance data manually.
6. Make the final refresh decision through human review.

The model score is used to allocate review effort. It does not automatically instruct the content team to edit a page.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Artifacts

The following artifacts support the research findings and should be generated or collected from the executed notebook:

1. **Sealed-test Average Precision comparison:** Bar chart comparing the rule baseline, Logistic Regression and Random Forest on the June test set.
2. **Random Forest feature importance:** Bar chart showing the relative importance of the nine input features in the Random Forest model.
3. **Sealed-test performance table:** ROC AUC, Average Precision, Precision@50, Precision@200 and Precision@1000 for all three methods.
4. **Development cross-validation table:** Development ROC AUC, Average Precision and Precision@50 for all three methods.
5. **Trend-bin analysis:** Table showing decline rates across bins of the trend ratio.
6. **Top-ranked page profile:** A summary of the characteristics of the highest-ranked pages, including their recent impressions and trend ratio.
7. **Leakage audit:** Evidence of the deliberate-leak test and the validation checks used to avoid leakage.
8. **Ranked recommendation output:** Example of the highest-risk pages with their pseudonymized content identifier, model score, reason code, recent impressions and trend ratio.

These artifacts provide evidence for model comparison, feature analysis, validation, leakage checks and practical decision-support recommendations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
